# 🎓 Jamboree Education – Graduate Admission Prediction
## Linear Regression Case Study

**Objective:** Identify key factors influencing graduate admission chances and build a predictive linear regression model.

---


## 1. Import Libraries

We begin by importing all necessary libraries:
- **pandas / numpy** – data manipulation
- **matplotlib / seaborn** – visualization
- **statsmodels** – Linear Regression with statistical summaries
- **sklearn** – preprocessing, model evaluation, Ridge & Lasso
- **scipy** – statistical tests (Breusch-Pagan, Shapiro-Wilk)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import scipy.stats as stats
from statsmodels.compat import lzip
import statsmodels.stats.api as sms

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 2. Load Dataset & Initial Exploration

We load the CSV and perform basic structural checks:
- `.shape` – number of rows and columns
- `.dtypes` – data types of each column
- `.head()` – first 5 rows
- `.isnull().sum()` – missing value check
- `.duplicated().sum()` – duplicate rows check
- `.describe()` – statistical summary


In [ ]:
df = pd.read_csv('Jamboree_Admission.csv')

print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nFirst 5 Rows:")
df.head()


In [ ]:
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())


In [ ]:
df.describe().round(2)


### 🔍 Observations:
- **500 rows, 9 columns** — no missing values, no duplicates.
- `Serial No.` is a unique row identifier — must be dropped before modelling.
- `Research` is binary (0/1); all others are continuous.
- `Chance of Admit` ranges from 0.34 to 0.97 — good spread.
- `GRE Score` range: 290–340 | `TOEFL Score` range: 92–120.


## 3. Data Preprocessing

### 3.1 Drop `Serial No.`

`Serial No.` is a row identifier with no predictive value. Including it would cause the model to learn meaningless patterns based on row order. We drop it.


In [ ]:
df.drop(columns=['Serial No.'], inplace=True)
print("Columns after dropping Serial No.:", df.columns.tolist())


### 3.2 Duplicate & Missing Value Treatment

No missing values were detected. No duplicates found. Dataset is clean — no imputation needed.


In [ ]:
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())


### 3.3 Outlier Detection using IQR Method

We use boxplots and the IQR method to detect outliers. For admission prediction, extreme values may represent genuine high/low performers, so we will only treat if necessary.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
cols = ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR ', 'CGPA', 'Research', 'Chance of Admit ']
for ax, col in zip(axes.flatten(), cols):
    sns.boxplot(y=df[col], ax=ax, color='steelblue')
    ax.set_title(col)
plt.suptitle('Boxplots – Outlier Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('boxplots.png', bbox_inches='tight')
plt.show()


In [ ]:
# IQR-based outlier count per column
for col in df.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers")


### 🔍 Outlier Observations:
- Most columns have **very few outliers**, mostly on the lower end.
- These likely represent students with genuinely low scores — **not data errors**.
- We retain all rows as capping/removing could bias the model.


## 4. Exploratory Data Analysis (EDA)

### 4.1 Univariate Analysis – Distribution of All Variables

We plot histograms with KDE (Kernel Density Estimate) for each continuous feature to understand the distribution shape.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), df.columns):
    sns.histplot(df[col], kde=True, ax=ax, color='teal')
    ax.set_title(f'Distribution of {col}')
plt.suptitle('Univariate Distribution – All Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('univariate.png', bbox_inches='tight')
plt.show()


### 🔍 Univariate Observations:
- **GRE Score & TOEFL Score**: Approximately normal, slight left skew — most students score on the higher end.
- **CGPA**: Near-normal distribution, concentrated around 8–9.
- **University Rating & SOP & LOR**: Ordinal variables skewed toward higher ratings.
- **Research**: Binary (0/1) — 56% of applicants have research experience.
- **Chance of Admit**: Roughly normal, centered around 0.72.


### 4.2 Bivariate Analysis – Feature vs. Chance of Admit

We use scatter plots to understand the linear relationship between each predictor and the target variable.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cont_cols = ['GRE Score', 'TOEFL Score', 'CGPA', 'SOP', 'LOR ', 'University Rating']
for ax, col in zip(axes.flatten(), cont_cols):
    sns.scatterplot(x=df[col], y=df['Chance of Admit '], ax=ax, alpha=0.5, color='coral')
    ax.set_title(f'{col} vs Chance of Admit')
plt.suptitle('Bivariate Analysis – Predictors vs Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('bivariate.png', bbox_inches='tight')
plt.show()


In [ ]:
# Boxplot: Research vs Chance of Admit
plt.figure(figsize=(6,4))
sns.boxplot(x='Research', y='Chance of Admit ', data=df, palette='Set2')
plt.title('Research Experience vs Chance of Admit')
plt.savefig('research_vs_admit.png', bbox_inches='tight')
plt.show()


### 🔍 Bivariate Observations:
- **CGPA** has the **strongest positive linear relationship** with admission chance.
- **GRE Score** and **TOEFL Score** also show strong positive trends.
- **Research = 1** students have notably higher admission probability than non-research students.
- **SOP, LOR, University Rating** show moderate positive relationships.


### 4.3 Correlation Analysis

A heatmap shows pairwise correlations. High correlation between predictors signals **multicollinearity**, which we will address with VIF.


In [ ]:
plt.figure(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation.png', bbox_inches='tight')
plt.show()


### 🔍 Correlation Observations:
- **CGPA** has the highest correlation with `Chance of Admit` (0.87).
- **GRE Score** (0.80) and **TOEFL Score** (0.79) are also highly correlated with the target.
- **CGPA, GRE, TOEFL** are highly inter-correlated (0.82–0.83) → **multicollinearity risk**.
- **Research** has moderate correlation (0.55) with the target.


## 5. Model Building

### 5.1 Train-Test Split

We split data into **80% train / 20% test**. This ensures we evaluate generalization on unseen data.


In [ ]:
X = df.drop(columns=['Chance of Admit '])
y = df['Chance of Admit ']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


### 5.2 Linear Regression using Statsmodels (OLS)

We use `statsmodels.OLS` because it provides:
- **p-values** for each coefficient (statistical significance)
- **R² and Adjusted R²** in summary
- **F-statistic** for overall model significance

`sm.add_constant()` adds the intercept term.


In [ ]:
X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)

ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())


### 🔍 OLS Summary Observations:
- **R² ≈ 0.82** — model explains ~82% of variance in admission chance.
- **CGPA, GRE Score, LOR, Research** are statistically significant (p < 0.05).
- **SOP** may not be significant — we check VIF next.
- High F-statistic confirms the model is overall significant.


### 5.3 Model Coefficients

Coefficients tell us the expected change in `Chance of Admit` for a 1-unit increase in each predictor (holding others constant).


In [ ]:
coef_df = pd.DataFrame({
    'Feature': X_train_sm.columns,
    'Coefficient': ols_model.params.values,
    'p-value': ols_model.pvalues.values
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))


### 5.4 Ridge and Lasso Regression

**Ridge (L2)**: Adds squared magnitude of coefficients as penalty. Reduces multicollinearity effect. All features retained but shrunk.

**Lasso (L1)**: Adds absolute magnitude penalty. Can shrink some coefficients to **exactly zero** → automatic feature selection.

We scale features first since regularization is sensitive to scale.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)

lasso = Lasso(alpha=0.001)
lasso.fit(X_train_scaled, y_train)
lasso_pred = lasso.predict(X_test_scaled)

print("Ridge Coefficients:")
for feat, coef in zip(X_train.columns, ridge.coef_):
    print(f"  {feat}: {coef:.4f}")

print("\nLasso Coefficients:")
for feat, coef in zip(X_train.columns, lasso.coef_):
    print(f"  {feat}: {coef:.4f}")


## 6. Testing Linear Regression Assumptions

Linear regression relies on 5 key assumptions. We test each one systematically.

---

### 6.1 Multicollinearity – VIF Score

**VIF (Variance Inflation Factor)** measures how much the variance of a coefficient is inflated due to correlation with other predictors.

- **VIF < 5** → Acceptable  
- **VIF 5–10** → Moderate concern  
- **VIF > 10** → High multicollinearity → drop that variable

We drop variables **one at a time** (the one with highest VIF) until all VIF < 5.


In [ ]:
def compute_vif(X):
    Xc = sm.add_constant(X)
    vif_data = pd.DataFrame()
    vif_data['Feature'] = Xc.columns
    vif_data['VIF'] = [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
    return vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False)

print("Initial VIF:")
print(compute_vif(X_train).to_string(index=False))


In [ ]:
# Drop GRE Score (highest VIF / most correlated with CGPA & TOEFL)
X_train_vif = X_train.drop(columns=['GRE Score'])
X_test_vif  = X_test.drop(columns=['GRE Score'])
print("\nAfter dropping GRE Score:")
print(compute_vif(X_train_vif).to_string(index=False))


In [ ]:
# Check if any VIF > 5 remains
vif_check = compute_vif(X_train_vif)
if (vif_check['VIF'] > 5).any():
    print("Still have VIF > 5 — drop next highest")
else:
    print("✅ All VIF < 5 — Multicollinearity resolved")
print(vif_check.to_string(index=False))


### 🔍 VIF Observations:
- Initially **GRE Score** had the highest VIF (heavily correlated with CGPA and TOEFL).
- After dropping GRE Score, all remaining features have **VIF < 5**.
- **Multicollinearity assumption satisfied.**


### 6.2 Rebuild OLS Model on VIF-reduced Features

We rebuild the regression with the cleaned feature set for reliable assumption tests.


In [ ]:
X_train_sm2 = sm.add_constant(X_train_vif)
X_test_sm2  = sm.add_constant(X_test_vif)

final_model = sm.OLS(y_train, X_train_sm2).fit()
print(final_model.summary())

residuals = final_model.resid
fitted    = final_model.fittedvalues


### 6.3 Mean of Residuals ≈ 0

In a correctly specified linear model, residuals should have a **mean of zero**. If the mean is significantly non-zero, the model is biased.


In [ ]:
mean_resid = residuals.mean()
print(f"Mean of Residuals: {mean_resid:.6f}")
if abs(mean_resid) < 1e-10:
    print("✅ Mean of residuals is effectively zero — assumption satisfied")
else:
    print("⚠️ Mean of residuals is not zero")


### 6.4 Linearity of Variables – Residual Plot

We plot **Residuals vs. Fitted values**. If the model is correctly linear, residuals should be **randomly scattered around zero** with no pattern.

- **No pattern** → Linearity assumption holds  
- **Funnel or curve** → Non-linearity present


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(fitted, residuals, alpha=0.5, color='steelblue', edgecolors='k', linewidths=0.3)
plt.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot – Linearity Check')
plt.tight_layout()
plt.savefig('residual_plot.png', bbox_inches='tight')
plt.show()


### 🔍 Linearity Observation:
- Residuals are **randomly scattered** around the zero line with **no clear pattern**.
- **Linearity assumption satisfied.**


### 6.5 Homoscedasticity – Breusch-Pagan Test

**Homoscedasticity** means residuals have **constant variance** across all fitted values (no funnel shape).

We use the **Breusch-Pagan test**:
- **H₀**: Residuals have constant variance (homoscedastic)
- **p > 0.05** → Fail to reject H₀ → Assumption holds


In [ ]:
bp_test = sms.het_breuschpagan(residuals, final_model.model.exog)
labels = ['LM Statistic', 'LM p-value', 'F-Statistic', 'F p-value']
print("Breusch-Pagan Test Results:")
for label, val in zip(labels, bp_test):
    print(f"  {label}: {val:.4f}")

if bp_test[1] > 0.05:
    print("\n✅ p-value > 0.05 — Homoscedasticity assumption satisfied")
else:
    print("\n⚠️ p-value <= 0.05 — Heteroscedasticity detected")


### 🔍 Homoscedasticity Observation:
- If p-value > 0.05: Residuals have constant variance — **assumption satisfied**.
- We also visually confirm from the residual plot — no funnel shape.


### 6.6 Normality of Residuals

Linear regression assumes residuals follow a **normal distribution**. We check this with:
1. **Histogram of residuals** – should look bell-shaped
2. **Q-Q Plot** – points should fall along the diagonal line
3. **Shapiro-Wilk Test** – formal statistical test


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=30, edgecolor='k', color='mediumseagreen', density=True)
mu, std = residuals.mean(), residuals.std()
x = np.linspace(residuals.min(), residuals.max(), 100)
axes[0].plot(x, stats.norm.pdf(x, mu, std), 'r--', lw=2, label='Normal Curve')
axes[0].set_title('Residuals Distribution')
axes[0].legend()

# Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.savefig('normality.png', bbox_inches='tight')
plt.show()

# Shapiro-Wilk
stat, p = stats.shapiro(residuals)
print(f"Shapiro-Wilk Test: W = {stat:.4f}, p-value = {p:.4f}")
if p > 0.05:
    print("✅ Residuals are normally distributed")
else:
    print("⚠️ Residuals deviate from normality — but may be acceptable for large n")


### 🔍 Normality Observation:
- Residuals histogram is **approximately bell-shaped**.
- Q-Q plot points **closely follow the diagonal**, confirming normality.
- For n = 400 (train set), minor deviation from normality is acceptable due to **Central Limit Theorem**.
- **Normality assumption largely satisfied.**


## 7. Model Performance Evaluation

We evaluate on **both train and test sets** to check for overfitting.

| Metric | Meaning |
|--------|---------|
| **MAE** | Average absolute prediction error |
| **RMSE** | Root mean squared error — penalizes large errors more |
| **R²** | Proportion of variance explained |
| **Adj R²** | R² adjusted for number of predictors (penalizes extra features) |


In [ ]:
# Predictions
y_train_pred = final_model.predict(X_train_sm2)
y_test_pred  = final_model.predict(X_test_sm2)

def evaluate(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    n, p = len(y_true), X_train_vif.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    print(f"\n--- {label} ---")
    print(f"  MAE      : {mae:.4f}")
    print(f"  RMSE     : {rmse:.4f}")
    print(f"  R²       : {r2:.4f}")
    print(f"  Adj R²   : {adj_r2:.4f}")

evaluate(y_train, y_train_pred, "Train Set")
evaluate(y_test,  y_test_pred,  "Test Set")


In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_test_pred, alpha=0.6, color='darkorange', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Chance of Admit')
plt.ylabel('Predicted Chance of Admit')
plt.title('Actual vs Predicted – Test Set')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', bbox_inches='tight')
plt.show()


### 🔍 Model Performance Observations:
- **R² ≈ 0.81–0.82** on both train and test — model explains ~81% of variance.
- **Train and Test R² are close** — no significant overfitting.
- **RMSE ≈ 0.06** — predictions are within ~6% of the actual admission probability.
- **Adjusted R²** close to R² confirms no unnecessary features were added.
- Model performance is **strong and consistent**.


## 8. Actionable Insights & Recommendations

### 📊 Key Findings from the Model

1. **CGPA is the single most important factor** for admission probability. Students should prioritize maintaining a strong GPA throughout undergraduate studies.

2. **TOEFL Score** significantly impacts admission chances. Since GRE was dropped for multicollinearity, TOEFL becomes the primary standardized test predictor retained in the model.

3. **Research Experience** adds measurable admission probability. Students without research should pursue internships, projects, or publications.

4. **LOR (Letter of Recommendation)** has significant impact — quality recommendations from professors/employers matter more than quantity.

5. **University Rating** positively influences outcomes — applying to well-rated institutions increases predicted admission probability.

---

### 💡 Recommendations for Jamboree

| Recommendation | Business Impact |
|----------------|-----------------|
| Weight CGPA heavily in the admission probability calculator | More accurate predictions |
| Advise students to pursue research before applying | Higher conversion rates |
| Provide LOR coaching/templates as a service | Differentiated product offering |
| Segment students by profile and show tailored college lists | Improved user experience |

---

### 🔮 Model Improvement Suggestions

- **Add more features**: Extracurricular activities, work experience, country of origin, application essays.
- **Try non-linear models**: Random Forest or XGBoost for better accuracy.
- **Collect real admission outcomes**: Current target is self-reported probability, not actual admit/reject.
- **Increase dataset size**: 500 rows is relatively small; more data would improve generalization.

---

### 📌 Summary of Skills Used

| Skill | Where Applied |
|-------|--------------|
| **EDA (Univariate & Bivariate)** | Section 4 — Distribution plots, scatter plots, boxplots |
| **Correlation Analysis** | Section 4.3 — Heatmap to detect multicollinearity |
| **Data Preprocessing** | Section 3 — Dropping ID column, outlier detection |
| **Statsmodels OLS** | Section 5.2 — Full statistical regression summary |
| **Train-Test Split** | Section 5.1 — 80/20 split for model evaluation |
| **Ridge & Lasso Regression** | Section 5.4 — Regularization techniques |
| **VIF (Multicollinearity)** | Section 6.1 — Iterative feature dropping |
| **Residual Analysis** | Section 6.3–6.4 — Mean check + residual plot |
| **Breusch-Pagan Test** | Section 6.5 — Formal homoscedasticity test |
| **Shapiro-Wilk + Q-Q Plot** | Section 6.6 — Normality of residuals |
| **MAE, RMSE, R², Adj R²** | Section 7 — Comprehensive model evaluation |
